In [ ]:
from pathlib import Path
import os, sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/softplus_controller.py").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


In [ ]:
from src.environments.Environment import *
from src.agents.DDPG import *
from src.models.NN_Module import *
from configs.config import Config

import numpy as np
from numpy import linalg as LA
from tqdm import tqdm
import torch
import matplotlib.pyplot as plt

from loguru import logger
from scipy.io import loadmat

### a simple logger
logger.remove()
logger.add(sys.stderr, level='DEBUG')

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
logger.info(f"Using {device} device")

NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_axis_title": 28,
    "font_legend": 28,
    "dpi": 300
}


In [ ]:
p = loadmat('data/aggr_p.mat')
load_p = p['p']
q = loadmat('data/aggr_q.mat')
load_q = q['q']
PV_p = loadmat('data/PV.mat')
PV_p =np.resize(PV_p['actual_PV_profile'],(load_q.shape[0],1))


# load_p = np.repeat(load_p,6)
# load_q = np.repeat(load_q,6)
# PV_p = np.repeat(PV_p,6)


In [ ]:
env_seed = 10        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l 6l

agent_num = 5
agent_policy_net = []
safe_agent_net = []

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
# state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
state, topology, senario = env.reset(seed=env_seed)             #not change
topology = torch.tensor(topology).unsqueeze(0)
# env.step_load([0,0,0,0,0], load_p[8000], load_q[8000], PV_p[8000])
pf_res_plotly(pp_net)

pv_status = env.get_pv_status()
logger.info(f"PV status: {pv_status}")

logger.info(len(env.injection_bus))

result = env.pv_node_change(action='add', p_mw=PV_p[8000]*0.35)
logger.info(f"PV node change result: {result}")

# result = env.pv_node_change(action='disconnect')
# logger.info(f"PV node change result: {result}")

# pv_status = env.get_pv_status()
# logger.info(f"PV status: {pv_status}")

env.pv_node_reset()
env.reset0()

pf_res_plotly(pp_net)
print("---")


In [ ]:
import os
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_56bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-09/Step_250_Seed_23_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

In [ ]:
def run_simulation(p, q, pv_p, topology_change_step=7000):
    """
    Run simulation and collect data
    
    Parameters:
    -----------
    p : array
        Active power data
    q : array
        Reactive power data
    pv_p : array
        PV power data
    topology_change_step : int
        Step at which to add new PV node
    
    Returns:
    --------
    dict : Dictionary containing all simulation results
    """
    from tqdm import tqdm
    import numpy as np
    
    results = {
        'topology_change_step': topology_change_step,
        'agent_num': agent_num,
        'injection_bus': injection_bus.copy()
    }
    
    # First simulation: zero action control
    print('Running simulation with zero action control...')
    env.pv_node_reset()
    state = env.reset0()
    episode_reward = 0
    last_action = np.zeros((agent_num, 1))
    action_list = []
    state_list = []
    state_list.append(state)
    
    for step in tqdm(range(p.shape[0])):
        if step == topology_change_step:
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        action = np.zeros((agent_num, 1))
        next_state, topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        action_list.append(action)
        state_list.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    results['no_control'] = {
        'action_list': action_list,
        'state_list': np.array(state_list),
        'p': p[:len(action_list)],
        'q': q[:len(action_list)],
        'pv_p': pv_p[:len(action_list)]
    }
    
    # Second simulation: using RLCFT
    print('\nRunning simulation with RLCFT control...')
    env.pv_node_reset()
    state = env.reset0()
    topology = torch.cuda.FloatTensor(np.array(env.topology_init)).unsqueeze(0)
    last_action = np.zeros((agent_num, 1))
    action_list_RLCFT = []
    state_list_RLCFT = []
    state_list_RLCFT.append(state)
    
    for step in tqdm(range(p.shape[0])):
        if step == topology_change_step:
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 1.0* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        # PI policy    
        action = last_action - np.asarray(action)

        # Execute action
        next_state, topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    
        action_list_RLCFT.append(action)
        state_list_RLCFT.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    results['rlcft_control'] = {
        'action_list': action_list_RLCFT,
        'state_list': np.array(state_list_RLCFT)
    }
    
    print("\nSimulation completed!")
    return results

# Run simulation and save results
simulation_data = run_simulation(load_p, load_q, PV_p, topology_change_step=7000)

In [ ]:
import pickle, gzip, torch, copy, os

def save_sim_results(results, file_path=os.path.join(Config.data_path, 'cache', 'notebook_results', 'realworld_results.pkl.gz')):
    """
    把 run_imperfect_info_simulation / run_elementwise_info_simulation
    的返回值 results 保存到硬盘
    """
    # 深拷贝，避免对原数据造成副作用
    res = copy.deepcopy(results)

    # 把所有 torch.Tensor -> cpu -> numpy，确保 pickle 可用
    def to_cpu_ndarray(x):
        if isinstance(x, torch.Tensor):
            return x.cpu().numpy()
        return x

    for key in res.keys():
        if isinstance(res[key], dict):
            for subk in res[key]:
                if isinstance(res[key][subk], list):
                    res[key][subk] = [to_cpu_ndarray(v) for v in res[key][subk]]
        res[key] = to_cpu_ndarray(res[key])

    # 写文件（gzip 压缩）
    os.makedirs(os.path.dirname(file_path) or '.', exist_ok=True)
    with gzip.open(file_path, 'wb') as f:
        pickle.dump(res, f)

    print(f'✔ Results saved to {os.path.abspath(file_path)}')

def load_sim_results(file_path=os.path.join(Config.data_path, 'cache', 'notebook_results', 'realworld_results.pkl.gz')):
    """
    读取保存好的 results
    """
    with gzip.open(file_path, 'rb') as f:
        results = pickle.load(f)
    print(f'✔ Results loaded from {os.path.abspath(file_path)}')
    return results

# 保存不完美信息仿真结果
save_sim_results(simulation_data, file_path=os.path.join(Config.data_path, 'cache', 'notebook_results', 'realworld_results.pkl.gz'))

In [ ]:
from pathlib import Path
REAL_WORLD_OUTPUT_DIR = Path(Config.data_path) / 'images' / 'real_world_evaluation'
REAL_WORLD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_axis_title": 28,
    "font_legend": 28,
    "dpi": 300
}

def plot_simulation_results(sim_data, save_fig=True):
    """
    Plot simulation results
    
    Parameters:
    -----------
    sim_data : dict
        Simulation results from run_simulation()
    save_fig : bool
        Whether to save the figure
    
    Returns:
    --------
    fig : plotly figure object
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import numpy as np
    
    # Extract data
    topology_change_step = sim_data['topology_change_step']
    agent_num = sim_data['agent_num']
    injection_bus = sim_data['injection_bus']
    
    no_control_data = sim_data['no_control']
    rlcft_data = sim_data['rlcft_control']
    
    # Create figure with subplots
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Power Profiles", "Bus Voltage without Control", "Bus Voltage with RLC-FT"),
        horizontal_spacing=0.08,
    )
    for annotation in fig.layout.annotations:
        annotation.font.size = NATURE_CONFIG['font_title']
    
    x_values = list(range(len(no_control_data['action_list'])))
    
    
    # First subplot: power curves
    fig.add_trace(
        go.Scatter(x=x_values, y=no_control_data['p'].flatten(), mode='lines', name='Active Load', 
                  line=dict(width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=x_values, y=no_control_data['q'].flatten(), mode='lines', name='Reactive Load', 
                  line=dict(width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=x_values, y=no_control_data['pv_p'].flatten(), mode='lines', name='Solar', 
                  line=dict(width=2)),
        row=1, col=1
    )
    
    # Second subplot: bus voltage without control
    state_list_array = no_control_data['state_list']
    for i in range(agent_num):
        fig.add_trace(
            go.Scatter(
                x=x_values, 
                y=state_list_array[:len(no_control_data['action_list']), i], 
                mode='lines', 
                name=f'Bus {injection_bus[i]+1}', 
                line=dict(width=2)
            ),
            row=1, col=2
        )
    
    # Add voltage limit lines
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[0.95]*len(no_control_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            name='Lower Limit',
            showlegend=True
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[1.05]*len(no_control_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            name='Upper Limit',
            showlegend=True
        ),
        row=1, col=2
    )
    
    # Third subplot: bus voltage with RLCFT
    state_list_RLCFT_array = rlcft_data['state_list']
    for i in range(agent_num):
        fig.add_trace(
            go.Scatter(
                x=x_values, 
                y=state_list_RLCFT_array[:len(rlcft_data['action_list']), i], 
                mode='lines', 
                name=f'Bus {injection_bus[i]+1} (RLC-FT)', 
                line=dict(width=2),
                showlegend=False
            ),
            row=1, col=3
        )
    
    # Add voltage limit lines
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[0.95]*len(rlcft_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            showlegend=False
        ),
        row=1, col=3
    )
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[1.05]*len(rlcft_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            showlegend=False
        ),
        row=1, col=3
    )
    
    # Add vertical lines for topology change
    for col in [2, 3]:
        fig.add_vline(
            x=topology_change_step,
            line_width=2,
            line_dash="dash",
            line_color="#FF0000",
            opacity=0.5,
            row=1, col=col
        )
    
    # Add shaded region to highlight the change
    fig.add_shape(
        type="rect",
        x0=topology_change_step-100, x1=topology_change_step+100,
        y0=0.92, y1=1.15,
        fillcolor="red",
        opacity=0.1,
        line_width=10,
        row=1, col=2
    )
    
    fig.add_shape(
        type="rect",
        x0=topology_change_step-100, x1=topology_change_step+100,
        y0=0.92, y1=1.10,
        fillcolor="red",
        opacity=0.1,
        line_width=10,
        row=1, col=3
    )
    
    # Add text annotation
    fig.add_annotation(
        x=topology_change_step,
        y=1.13,
        text="New PV Station Added",
        showarrow=True,
        arrowhead=2,
        arrowsize=2,
        arrowwidth=1,
        arrowcolor="red",
        ax=0,
        ay=-30,
        font=dict(size=NATURE_CONFIG['font_base']-2, color="red"),
        xref="x2",
        yref="y2"
    )
    
    fig.add_annotation(
        x=topology_change_step,
        y=1.08,
        text="New PV Station Added",
        showarrow=True,
        arrowhead=2,
        arrowsize=2,
        arrowwidth=1,
        arrowcolor="red",
        ax=0,
        ay=-30,
        font=dict(size=NATURE_CONFIG['font_base']-2, color="red"),
        xref="x3",
        yref="y3"
    )

    # Label for the first panel (Power Profiles)
    fig.add_annotation(
        text="<b>a</b>", 
        x=-0.02, y=1.08, # Adjust as needed; x might be near the y-axis of the first plot
        xref="paper", yref="paper", 
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"), # Match your other (a), (b) labels
        align="left"
    )
    fig.add_annotation(
        text="<b>b</b>", 
        x=0.35, y=1.08, # EXAMPLE X - ADJUST THIS based on your subplot spacing
        xref="paper", yref="paper", 
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"),
        align="left"
    )
    fig.add_annotation(
        text="<b>c</b>", 
        x=0.71, y=1.08, # EXAMPLE X - ADJUST THIS
        xref="paper", yref="paper", 
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"),
        align="left"
    )
    
    # Set time axis ticks
    time_labels = ['00:00', '06:00', '12:00', '18:00', '24:00']
    time_ticks = list(range(0, len(no_control_data['action_list']), 21600))
    if len(time_ticks) < len(time_labels):
        time_ticks = [0]
        if len(no_control_data['action_list']) > 1:
            step_size = len(no_control_data['action_list']) // 4
            time_ticks = [0, step_size, 2*step_size, 3*step_size, min(4*step_size, len(no_control_data['action_list'])-1)]
    
    # Update layout and axes
    fig.update_layout(
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        height=NATURE_CONFIG["height"],
        width=NATURE_CONFIG["width"],
        margin=dict(l=60, r=30, t=60, b=100),
        template="plotly_white",
        font=dict(size=NATURE_CONFIG["font_base"], family="Arial"),
    )
    
    # Update X axes
    for i in range(1, 4):
        fig.update_xaxes(
            title_text="Time (Hour)",
            tickvals=time_ticks,
            ticktext=time_labels,
            title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
            row=1, col=i
        )
    
    # Update Y axes
    fig.update_yaxes(
        title_text="Power (MW/MVar)", 
        title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
        row=1, col=1
    )
    

    fig.update_yaxes(
        title_text="Bus voltage (p.u.)", 
        range=[0.92, 1.10],
        tickvals=[0.95, 1.00, 1.05, 1.10],
        ticktext=['0.95', '1.00', '1.05', '1.10'],
        title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
        row=1, col=3
    )

    fig.update_yaxes(
            title_text="Bus voltage (p.u.)", 
            range=[0.92, 1.15],
            tickvals=[0.95, 1.00, 1.05, 1.10],
            ticktext=['0.95', '1.00', '1.05', '1.10'],
            title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
            row=1, col=2
        )
    
    # Display figure
    fig.show()
    
    # Save figure if requested
    if save_fig:
        fig.write_html(REAL_WORLD_OUTPUT_DIR / "realdata_interactive.html")
        fig.write_image(REAL_WORLD_OUTPUT_DIR / "realdata.pdf", scale=1.5)
        fig.write_image(REAL_WORLD_OUTPUT_DIR / "realdata.png", scale=1.5)
        print("Figure saved as 'realdata_interactive.html' and 'realdata.pdf'")
    
    return fig

# Plot the results
fig = plot_simulation_results(simulation_data, save_fig=False)

In [ ]:
def run_imperfect_info_simulation(p, q, pv_p, topology_change_step=7000, loss_rate=0.3, max_delay=10):
    """
    对比完美信息和不完美信息(30%丢失+随机延迟)下的控制效果
    """
    import numpy as np
    from tqdm import tqdm
    import torch
    import random
    
    results = {
        'topology_change_step': topology_change_step,
        'agent_num': agent_num,
        'injection_bus': injection_bus.copy(),
        'loss_rate': loss_rate,
        'max_delay': max_delay
    }
    
    # 第一个仿真：完美信息的RLCFT控制
    print('Running simulation with perfect information RLCFT...')
    env.pv_node_reset()
    state = env.reset0()
    topology = torch.cuda.FloatTensor(np.array(env.topology_init)).unsqueeze(0)
    last_action = np.zeros((agent_num, 1))
    action_list_perfect = []
    state_list_perfect = []
    state_list_perfect.append(state)
    
    for step in tqdm(range(p.shape[0])):
        if step == topology_change_step:
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 1.0 * action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)
        
        # PI policy    
        action = last_action - np.asarray(action)
        
        # Execute action
        next_state, new_topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        topology = torch.cuda.FloatTensor(new_topology).unsqueeze(0)
        
        action_list_perfect.append(action)
        state_list_perfect.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    results['perfect_info'] = {
        'action_list': action_list_perfect,
        'state_list': np.array(state_list_perfect)
    }
    
    # 第二个仿真：不完美信息的RLCFT控制
    print(f'\nRunning simulation with imperfect information')
    env.pv_node_reset()
    state = env.reset0()

    # 为每个agent维护独立的信息状态
    agent_info_states = []
    for i in range(agent_num):
        agent_info_states.append({
            'local_topology': torch.cuda.FloatTensor(np.array(env.topology_init)).unsqueeze(0).clone(),
            'pending_updates': []
        })

    last_action = np.zeros((agent_num, 1))
    action_list_imperfect = []
    state_list_imperfect = []
    state_list_imperfect.append(state)
    
    # 调试变量
    topology_update_log = []

    for step in tqdm(range(p.shape[0])):
        # 处理每个agent的延迟更新
        for i in range(agent_num):
            old_topology = agent_info_states[i]['local_topology'].clone()
            process_agent_updates(agent_info_states[i], step)
            # 检查是否有更新
            if not torch.equal(old_topology, agent_info_states[i]['local_topology']):
                topology_update_log.append(f"Step {step}: Agent {i} topology updated")
        
        # 检查是否有topology变化
        if step == topology_change_step:
            new_topology = env.topology_change(seed=1)
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        
        # 计算每个agent的action（基于各自的local_topology）
        action = []
        for i in range(agent_num):
            agent_topology = agent_info_states[i]['local_topology']
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), agent_topology)
            action_agent = 1.0 * action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)
        
        # PI policy    
        action = last_action - np.asarray(action)
        
        # Execute action - 这里会得到真实的当前topology
        next_state, current_true_topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        
        # 如果这一步有topology变化，模拟信息传输
        if step == topology_change_step:
            true_topology_tensor = torch.cuda.FloatTensor(current_true_topology).unsqueeze(0)
            print(f"Broadcasting topology change at step {step}")
            simulate_topology_broadcast(agent_info_states, true_topology_tensor, step, loss_rate, max_delay)
            
            # 调试：检查broadcast后各agent的状态
            for i, agent_state in enumerate(agent_info_states):
                print(f"Agent {i}: pending_updates = {len(agent_state['pending_updates'])}")
                if len(agent_state['pending_updates']) > 0:
                    print(f"  Next arrival: step {agent_state['pending_updates'][0][1]}")
        
        action_list_imperfect.append(action)
        state_list_imperfect.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    # 修复：将结果赋值移到循环外面
    results['imperfect_info'] = {
        'action_list': action_list_imperfect,
        'state_list': np.array(state_list_imperfect)
    }
    
    # 输出调试信息
    print(f"\nTopology update log: {len(topology_update_log)} updates")
    for log in topology_update_log[:10]:  # 只显示前10个
        print(log)
    
    # 检查最终状态
    print("\nFinal agent topology states:")
    for i, agent_state in enumerate(agent_info_states):
        print(f"Agent {i}: {len(agent_state['pending_updates'])} pending updates")
    
    print("\nSimulation completed!")
    return results


def simulate_topology_broadcast(agent_states, new_topology, current_step, loss_rate, max_delay):
    """
    模拟topology变化信息的广播
    """
    import random
    
    print(f"\nSimulating broadcast to {len(agent_states)} agents...")
    
    for i, agent_state in enumerate(agent_states):
        # 独立的信息丢失判断
        if random.random() > loss_rate:  # (1-loss_rate)概率接收
            # 添加随机延迟
            delay = random.randint(0, max_delay)
            arrival_step = current_step + delay
            
            print(f"Agent {i}: Receives update, delay = {delay} steps")
            
            if delay == 0:
                # 立即更新
                agent_state['local_topology'] = new_topology.clone()
                print(f"Agent {i}: Updated immediately")
            else:
                # 添加到延迟队列
                agent_state['pending_updates'].append((new_topology.clone(), arrival_step))
                print(f"Agent {i}: Update queued for step {arrival_step}")
        else:
            print(f"Agent {i}: Message lost")


def process_agent_updates(agent_state, current_step):
    """处理agent的延迟更新"""
    arrived_updates = []
    remaining_updates = []
    
    for topology, arrival_step in agent_state['pending_updates']:
        if arrival_step <= current_step:
            arrived_updates.append((topology, arrival_step))
        else:
            remaining_updates.append((topology, arrival_step))
    
    agent_state['pending_updates'] = remaining_updates
    
    # 如果有到达的更新，使用最新的
    if arrived_updates:
        arrived_updates.sort(key=lambda x: x[1])
        latest_topology, _ = arrived_updates[-1]
        agent_state['local_topology'] = latest_topology.clone()

# 运行不完美信息仿真
### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
# state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
state, topology, senario = env.reset(seed=env_seed)             #not change

imperfect_info_data = run_imperfect_info_simulation(load_p, load_q, PV_p, topology_change_step=7000, loss_rate=0.5, max_delay=100)

In [ ]:
import pickle, gzip, torch, copy, os

def save_sim_results(results, file_path=os.path.join(Config.data_path, 'cache', 'notebook_results', 'imperfect_results.pkl.gz')):
    """
    把 run_imperfect_info_simulation / run_elementwise_info_simulation
    的返回值 results 保存到硬盘
    """
    # 深拷贝，避免对原数据造成副作用
    res = copy.deepcopy(results)

    # 把所有 torch.Tensor -> cpu -> numpy，确保 pickle 可用
    def to_cpu_ndarray(x):
        if isinstance(x, torch.Tensor):
            return x.cpu().numpy()
        return x

    for key in res.keys():
        if isinstance(res[key], dict):
            for subk in res[key]:
                if isinstance(res[key][subk], list):
                    res[key][subk] = [to_cpu_ndarray(v) for v in res[key][subk]]
        res[key] = to_cpu_ndarray(res[key])

    # 写文件（gzip 压缩）
    os.makedirs(os.path.dirname(file_path) or '.', exist_ok=True)
    with gzip.open(file_path, 'wb') as f:
        pickle.dump(res, f)

    print(f'✔ Results saved to {os.path.abspath(file_path)}')

def load_sim_results(file_path=os.path.join(Config.data_path, 'cache', 'notebook_results', 'imperfect_results.pkl.gz')):
    """
    读取保存好的 results
    """
    if not os.path.exists(file_path):
        legacy = PROJECT_ROOT / 'data' / 'reference' / 'imperfect_results.pkl.gz'
        if legacy.is_file():
            file_path = legacy
    with gzip.open(file_path, 'rb') as f:
        results = pickle.load(f)
    print(f'✔ Results loaded from {os.path.abspath(file_path)}')
    return results

# 保存不完美信息仿真结果
save_sim_results(imperfect_info_data, file_path=os.path.join(Config.data_path, 'cache', 'notebook_results', 'imperfect_results.pkl.gz'))

In [ ]:
from pathlib import Path
REAL_WORLD_OUTPUT_DIR = Path(Config.data_path) / 'images' / 'real_world_evaluation'
REAL_WORLD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 加载不完美信息仿真结果
imperfect_info_data = load_sim_results()

def plot_imperfect_info_results_with_zoom(sim_data, save_fig=True, zoom_window=200):
    """
    Plot comparison between perfect and imperfect information scenarios with zoom-in view
    
    Parameters:
    -----------
    sim_data : dict
        Simulation results from run_imperfect_info_simulation()
    save_fig : bool
        Whether to save the figure
    zoom_window : int
        Number of steps before and after topology change to show in zoom view
    
    Returns:
    --------
    fig : plotly figure object
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import numpy as np
    
    # Extract data
    topology_change_step = sim_data['topology_change_step']
    agent_num = sim_data['agent_num']
    injection_bus = sim_data['injection_bus']
    loss_rate = sim_data.get('loss_rate', 0.3)
    max_delay = sim_data.get('max_delay', 10)
    
    perfect_data = sim_data['perfect_info']
    imperfect_data = sim_data['imperfect_info']
    
    # Create figure with 3 subplots
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            "Bus Voltage with Perfect Information", 
            f"Bus Voltage with Imperfect Information",
            "Zoom-in: Topology Change Region"
        ),
        horizontal_spacing=0.08,
        column_widths=[0.3, 0.3, 0.4]  # Give more space to the zoom plot
    )
    
    # Update subplot title font size
    for annotation in fig.layout.annotations:
        annotation.font.size = NATURE_CONFIG['font_title'] - 2
    
    x_values = list(range(len(perfect_data['action_list'])))
    
    # Define zoom region
    zoom_start = max(0, topology_change_step - zoom_window)
    zoom_end = min(len(perfect_data['action_list']), topology_change_step + zoom_window)
    zoom_x = list(range(zoom_start, zoom_end))
    
    # First subplot: Perfect Information
    perfect_state_array = perfect_data['state_list']
    for i in range(agent_num):
        fig.add_trace(
            go.Scatter(
                x=x_values, 
                y=perfect_state_array[:len(perfect_data['action_list']), i], 
                mode='lines', 
                name=f'Bus {injection_bus[i]+1}', 
                line=dict(width=2)
            ),
            row=1, col=1
        )
    
    # Add voltage limit lines for perfect info
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[0.95]*len(perfect_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            name='Lower Limit',
            showlegend=True
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[1.05]*len(perfect_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            name='Upper Limit',
            showlegend=True
        ),
        row=1, col=1
    )
    
    # Second subplot: Imperfect Information
    imperfect_state_array = imperfect_data['state_list']
    for i in range(agent_num):
        fig.add_trace(
            go.Scatter(
                x=x_values, 
                y=imperfect_state_array[:len(imperfect_data['action_list']), i], 
                mode='lines', 
                name=f'Bus {injection_bus[i]+1} (Imperfect)', 
                line=dict(width=2),
                showlegend=False  # Don't duplicate in legend
            ),
            row=1, col=2
        )
    
    # Add voltage limit lines for imperfect info
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[0.95]*len(imperfect_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            showlegend=False
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[1.05]*len(imperfect_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            showlegend=False
        ),
        row=1, col=2
    )
    
    # Third subplot: Zoom-in comparison
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']  # Default plotly colors
    
    for i in range(agent_num):
        # Perfect info (solid lines)
        fig.add_trace(
            go.Scatter(
                x=zoom_x,
                y=perfect_state_array[zoom_start:zoom_end, i],
                mode='lines',
                name=f'Bus {injection_bus[i]+1} (Perfect)',
                line=dict(width=3, color=colors[i % len(colors)]),
                showlegend=True
            ),
            row=1, col=3
        )
        
        # Imperfect info (dashed lines)
        fig.add_trace(
            go.Scatter(
                x=zoom_x,
                y=imperfect_state_array[zoom_start:zoom_end, i],
                mode='lines',
                name=f'Bus {injection_bus[i]+1} (Imperfect)',
                line=dict(width=3, color=colors[i % len(colors)], dash='dash'),
                showlegend=True
            ),
            row=1, col=3
        )
    
    # Add voltage limit lines for zoom plot
    fig.add_trace(
        go.Scatter(
            x=zoom_x, 
            y=[0.95]*(zoom_end-zoom_start), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dot'), 
            name='Voltage Limits',
            showlegend=False
        ),
        row=1, col=3
    )
    fig.add_trace(
        go.Scatter(
            x=zoom_x, 
            y=[1.05]*(zoom_end-zoom_start), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dot'), 
            showlegend=False
        ),
        row=1, col=3
    )
    
    # Add vertical lines for topology change
    for col in [1, 2, 3]:
        fig.add_vline(
            x=topology_change_step,
            line_width=3,
            line_dash="dash",
            line_color="#FF0000",
            opacity=0.8,
            row=1, col=col
        )
    
    # Add shaded regions to highlight the change (only for first two plots)
    y_range_max = 1.10
    y_range_min = 0.92
    
    for col in [1, 2]:
        fig.add_shape(
            type="rect",
            x0=topology_change_step-20, x1=topology_change_step+20,
            y0=y_range_min, y1=y_range_max,
            fillcolor="red",
            opacity=0.1,
            line_width=0,
            row=1, col=col
        )
    
    # Add text annotations for topology change
    for i, col in enumerate([1, 2, 3]):
        y_pos = 1.08 if col != 3 else (np.max([perfect_state_array[zoom_start:zoom_end, :].max(), 
                                               imperfect_state_array[zoom_start:zoom_end, :].max()]) + 0.01)
        fig.add_annotation(
            x=topology_change_step,
            y=y_pos,
            text="PV Added" if col == 3 else "New PV Station Added",
            showarrow=True,
            arrowhead=2,
            arrowsize=1.5,
            arrowwidth=1,
            arrowcolor="red",
            ax=0,
            ay=-25,
            font=dict(size=NATURE_CONFIG['font_base']-4, color="red"),
            xref=f"x{col}",
            yref=f"y{col}"
        )
    
    # Add subplot labels (a), (b), (c)
    labels = ['a', 'b', 'c']
    x_positions = [-0.03, 0.30, 0.69]
    for i, (label, x_pos) in enumerate(zip(labels, x_positions)):
        fig.add_annotation(
            text=f"<b>{label}</b>", 
            x=x_pos, y=1.08,
            xref="paper", yref="paper", 
            showarrow=False,
            font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"),
            align="left"
        )
    
    # Set time axis ticks for first two plots
    data_length = len(perfect_data['action_list'])
    time_ticks = [0, data_length//4, data_length//2, 3*data_length//4, data_length-1]
    time_labels = ['Start', '25%', '50%', '75%', 'End']
    
    # Set time axis ticks for zoom plot
    zoom_ticks = [zoom_start, topology_change_step-zoom_window//2, topology_change_step, 
                  topology_change_step+zoom_window//2, zoom_end-1]
    zoom_labels = [f'{topology_change_step-zoom_window}', f'{topology_change_step-zoom_window//2}', 
                   f'{topology_change_step}', f'{topology_change_step+zoom_window//2}', f'{zoom_end-1}']
    
    # Update layout
    fig.update_layout(
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.18,
            xanchor="center",
            x=0.5,
            font=dict(size=NATURE_CONFIG["font_legend"]-10)
        ),
        height=NATURE_CONFIG["height"],
        width=NATURE_CONFIG["width"] * 1.2,  # Wider to accommodate 3 plots
        margin=dict(l=60, r=40, t=80, b=140),
        template="plotly_white",
        font=dict(size=NATURE_CONFIG["font_base"], family="Arial"),
    )
    
    # Update X axes
    for i in range(1, 3):
        fig.update_xaxes(
            title_text="Time Steps",
            tickvals=time_ticks,
            ticktext=time_labels,
            title_font=dict(size=NATURE_CONFIG["font_axis_title"]-2),
            tickfont=dict(size=NATURE_CONFIG["font_axis"]-2),
            row=1, col=i
        )
    
    # Update X axis for zoom plot
    fig.update_xaxes(
        title_text="Time Steps (Zoomed)",
        tickvals=zoom_ticks,
        ticktext=zoom_labels,
        title_font=dict(size=NATURE_CONFIG["font_axis_title"]-2),
        tickfont=dict(size=NATURE_CONFIG["font_axis"]-2),
        row=1, col=3
    )
    
    # Update Y axes
    for i in range(1, 3):
        fig.update_yaxes(
            title_text="Bus voltage (p.u.)", 
            range=[0.92, 1.10],
            tickvals=[0.95, 1.00, 1.05, 1.10],
            ticktext=['0.95', '1.00', '1.05', '1.10'],
            title_font=dict(size=NATURE_CONFIG["font_axis_title"]-2),
            tickfont=dict(size=NATURE_CONFIG["font_axis"]-2),
            row=1, col=i
        )
    
    # Update Y axis for zoom plot - auto-scale based on data in zoom region
    zoom_y_min = min(perfect_state_array[zoom_start:zoom_end, :].min(), 
                     imperfect_state_array[zoom_start:zoom_end, :].min()) - 0.005
    zoom_y_max = max(perfect_state_array[zoom_start:zoom_end, :].max(), 
                     imperfect_state_array[zoom_start:zoom_end, :].max()) + 0.005
    
    fig.update_yaxes(
        title_text="Bus voltage (p.u.)", 
        range=[zoom_y_min, zoom_y_max],
        title_font=dict(size=NATURE_CONFIG["font_axis_title"]-2),
        tickfont=dict(size=NATURE_CONFIG["font_axis"]-2),
        row=1, col=3
    )
    
    # Display figure
    fig.show()
    
    # Save figure if requested
    if save_fig:
        filename_base = str(REAL_WORLD_OUTPUT_DIR / f"imperfect_info_zoom_loss{int(loss_rate*100)}_delay{max_delay}")
        fig.write_html(f"{filename_base}_interactive.html")
        fig.write_image(f"{filename_base}.pdf", scale=1.5)
        fig.write_image(f"{filename_base}.png", scale=1.5)
        print(f"Figure saved as '{filename_base}_interactive.html' and '{filename_base}.pdf'")
    
    return fig



# 使用示例
fig = plot_imperfect_info_results_with_zoom(imperfect_info_data, zoom_window=150)

In [ ]:
def run_simulation_cp(p, q, pv_p, topology_change_step=7000):
    """
    Run simulation and collect data
    
    Parameters:
    -----------
    p : array
        Active power data
    q : array
        Reactive power data
    pv_p : array
        PV power data
    topology_change_step : int
        Step at which to add new PV node
    
    Returns:
    --------
    dict : Dictionary containing all simulation results
    """
    from tqdm import tqdm
    import numpy as np
    
    results = {
        'topology_change_step': topology_change_step,
        'agent_num': agent_num,
        'injection_bus': injection_bus.copy()
    }
    
    # First simulation: linear controller
    print('Running simulation with linear control...')
    env.pv_node_reset()
    state = env.reset0()
    episode_reward = 0
    last_action = np.zeros((agent_num, 1))
    action_list = []
    state_list = []
    state_list.append(state)
    
    for step in tqdm(range(p.shape[0])):
        if step == topology_change_step:
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        
        state1 = np.asarray(state-env.vmax)
        state2 = np.asarray(env.vmin-state)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((agent_num,1))
        
        action = (last_action - 10*d_v)
        
        last_action = np.copy(action)

        next_state, topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        action_list.append(action)
        state_list.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    results['no_control'] = {
        'action_list': action_list,
        'state_list': np.array(state_list),
        'p': p[:len(action_list)],
        'q': q[:len(action_list)],
        'pv_p': pv_p[:len(action_list)]
    }
    
    # Second simulation: using RLCFT
    print('\nRunning simulation with RLCFT control...')
    env.pv_node_reset()
    state = env.reset0()
    topology = torch.cuda.FloatTensor(np.array(env.topology_init)).unsqueeze(0)
    last_action = np.zeros((agent_num, 1))
    action_list_RLCFT = []
    state_list_RLCFT = []
    state_list_RLCFT.append(state)
    
    for step in tqdm(range(p.shape[0])):
        if step == topology_change_step:
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 1.0* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        # PI policy    
        action = last_action - np.asarray(action)

        # Execute action
        next_state, topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    
        action_list_RLCFT.append(action)
        state_list_RLCFT.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    results['rlcft_control'] = {
        'action_list': action_list_RLCFT,
        'state_list': np.array(state_list_RLCFT)
    }
    
    print("\nSimulation completed!")
    return results

# Run simulation and save results
simulation_data_cp = run_simulation_cp(load_p, load_q, PV_p, topology_change_step=7000)

In [ ]:
from pathlib import Path
REAL_WORLD_OUTPUT_DIR = Path(Config.data_path) / 'images' / 'real_world_evaluation'
REAL_WORLD_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_axis_title": 28,
    "font_legend": 28,
    "dpi": 300
}

def plot_simulation_results(sim_data, save_fig=True):
    """
    Plot simulation results
    
    Parameters:
    -----------
    sim_data : dict
        Simulation results from run_simulation()
    save_fig : bool
        Whether to save the figure
    
    Returns:
    --------
    fig : plotly figure object
    """
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import numpy as np
    
    # Extract data
    topology_change_step = sim_data['topology_change_step']
    agent_num = sim_data['agent_num']
    injection_bus = sim_data['injection_bus']
    
    no_control_data = sim_data['no_control']
    rlcft_data = sim_data['rlcft_control']
    
    # Create figure with subplots
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=("Power Profiles", "Bus Voltage without Linear Control", "Bus Voltage with RLC-FT"),
        horizontal_spacing=0.08,
    )
    for annotation in fig.layout.annotations:
        annotation.font.size = NATURE_CONFIG['font_title']
    
    x_values = list(range(len(no_control_data['action_list'])))
    
    
    # First subplot: power curves
    fig.add_trace(
        go.Scatter(x=x_values, y=no_control_data['p'].flatten(), mode='lines', name='Active Load', 
                  line=dict(width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=x_values, y=no_control_data['q'].flatten(), mode='lines', name='Reactive Load', 
                  line=dict(width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=x_values, y=no_control_data['pv_p'].flatten(), mode='lines', name='Solar', 
                  line=dict(width=2)),
        row=1, col=1
    )
    
    # Second subplot: bus voltage without control
    state_list_array = no_control_data['state_list']
    for i in range(agent_num):
        fig.add_trace(
            go.Scatter(
                x=x_values, 
                y=state_list_array[:len(no_control_data['action_list']), i], 
                mode='lines', 
                name=f'Bus {injection_bus[i]+1}', 
                line=dict(width=2)
            ),
            row=1, col=2
        )
    
    # Add voltage limit lines
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[0.95]*len(no_control_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            name='Lower Limit',
            showlegend=True
        ),
        row=1, col=2
    )
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[1.05]*len(no_control_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            name='Upper Limit',
            showlegend=True
        ),
        row=1, col=2
    )
    
    # Third subplot: bus voltage with RLCFT
    state_list_RLCFT_array = rlcft_data['state_list']
    for i in range(agent_num):
        fig.add_trace(
            go.Scatter(
                x=x_values, 
                y=state_list_RLCFT_array[:len(rlcft_data['action_list']), i], 
                mode='lines', 
                name=f'Bus {injection_bus[i]+1} (RLC-FT)', 
                line=dict(width=2),
                showlegend=False
            ),
            row=1, col=3
        )
    
    # Add voltage limit lines
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[0.95]*len(rlcft_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            showlegend=False
        ),
        row=1, col=3
    )
    fig.add_trace(
        go.Scatter(
            x=x_values, 
            y=[1.05]*len(rlcft_data['action_list']), 
            mode='lines', 
            line=dict(color='black', width=1, dash='dash'), 
            showlegend=False
        ),
        row=1, col=3
    )
    
    # Add vertical lines for topology change
    for col in [2, 3]:
        fig.add_vline(
            x=topology_change_step,
            line_width=2,
            line_dash="dash",
            line_color="#FF0000",
            opacity=0.5,
            row=1, col=col
        )
    
    # Add shaded region to highlight the change
    fig.add_shape(
        type="rect",
        x0=topology_change_step-100, x1=topology_change_step+100,
        y0=0.92, y1=1.15,
        fillcolor="red",
        opacity=0.1,
        line_width=10,
        row=1, col=2
    )
    
    fig.add_shape(
        type="rect",
        x0=topology_change_step-100, x1=topology_change_step+100,
        y0=0.92, y1=1.10,
        fillcolor="red",
        opacity=0.1,
        line_width=10,
        row=1, col=3
    )
    
    # Add text annotation
    fig.add_annotation(
        x=topology_change_step,
        y=1.13,
        text="New PV Station Added",
        showarrow=True,
        arrowhead=2,
        arrowsize=2,
        arrowwidth=1,
        arrowcolor="red",
        ax=0,
        ay=-30,
        font=dict(size=NATURE_CONFIG['font_base']-2, color="red"),
        xref="x2",
        yref="y2"
    )
    
    fig.add_annotation(
        x=topology_change_step,
        y=1.08,
        text="New PV Station Added",
        showarrow=True,
        arrowhead=2,
        arrowsize=2,
        arrowwidth=1,
        arrowcolor="red",
        ax=0,
        ay=-30,
        font=dict(size=NATURE_CONFIG['font_base']-2, color="red"),
        xref="x3",
        yref="y3"
    )

    # Label for the first panel (Power Profiles)
    fig.add_annotation(
        text="<b>a</b>", 
        x=-0.02, y=1.08, # Adjust as needed; x might be near the y-axis of the first plot
        xref="paper", yref="paper", 
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"), # Match your other (a), (b) labels
        align="left"
    )
    fig.add_annotation(
        text="<b>b</b>", 
        x=0.35, y=1.08, # EXAMPLE X - ADJUST THIS based on your subplot spacing
        xref="paper", yref="paper", 
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"),
        align="left"
    )
    fig.add_annotation(
        text="<b>c</b>", 
        x=0.71, y=1.08, # EXAMPLE X - ADJUST THIS
        xref="paper", yref="paper", 
        showarrow=False,
        font=dict(size=NATURE_CONFIG["font_title"]+2, family="Arial Bold", color="black"),
        align="left"
    )
    
    # Set time axis ticks
    time_labels = ['00:00', '06:00', '12:00', '18:00', '24:00']
    time_ticks = list(range(0, len(no_control_data['action_list']), 21600))
    if len(time_ticks) < len(time_labels):
        time_ticks = [0]
        if len(no_control_data['action_list']) > 1:
            step_size = len(no_control_data['action_list']) // 4
            time_ticks = [0, step_size, 2*step_size, 3*step_size, min(4*step_size, len(no_control_data['action_list'])-1)]
    
    # Update layout and axes
    fig.update_layout(
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.2,
            xanchor="center",
            x=0.5
        ),
        height=NATURE_CONFIG["height"],
        width=NATURE_CONFIG["width"],
        margin=dict(l=60, r=30, t=60, b=100),
        template="plotly_white",
        font=dict(size=NATURE_CONFIG["font_base"], family="Arial"),
    )
    
    # Update X axes
    for i in range(1, 4):
        fig.update_xaxes(
            title_text="Time (Hour)",
            tickvals=time_ticks,
            ticktext=time_labels,
            title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
            row=1, col=i
        )
    
    # Update Y axes
    fig.update_yaxes(
        title_text="Power (MW/MVar)", 
        title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
        row=1, col=1
    )
    

    fig.update_yaxes(
        title_text="Bus voltage (p.u.)", 
        range=[0.92, 1.10],
        tickvals=[0.95, 1.00, 1.05, 1.10],
        ticktext=['0.95', '1.00', '1.05', '1.10'],
        title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
        row=1, col=3
    )

    fig.update_yaxes(
            title_text="Bus voltage (p.u.)", 
            range=[0.92, 1.15],
            tickvals=[0.95, 1.00, 1.05, 1.10],
            ticktext=['0.95', '1.00', '1.05', '1.10'],
            title_font=dict(size=NATURE_CONFIG["font_axis_title"]),
            row=1, col=2
        )
    
    # Display figure
    fig.show()
    
    # Save figure if requested
    if save_fig:
        fig.write_html(REAL_WORLD_OUTPUT_DIR / "realdata_interactive.html")
        fig.write_image(REAL_WORLD_OUTPUT_DIR / "realdata.pdf", scale=1.5)
        fig.write_image(REAL_WORLD_OUTPUT_DIR / "realdata.png", scale=1.5)
        print("Figure saved as 'realdata_interactive.html' and 'realdata.pdf'")
    
    return fig

# Plot the results
fig = plot_simulation_results(simulation_data_cp, save_fig=False)

In [ ]:
def run_simulation_safe(p, q, pv_p, topology_change_step=7000):
    """
    Run simulation and collect data
    
    Parameters:
    -----------
    p : array
        Active power data
    q : array
        Reactive power data
    pv_p : array
        PV power data
    topology_change_step : int
        Step at which to add new PV node
    
    Returns:
    --------
    dict : Dictionary containing all simulation results
    """
    from tqdm import tqdm
    import numpy as np
    
    results = {
        'topology_change_step': topology_change_step,
        'agent_num': agent_num,
        'injection_bus': injection_bus.copy()
    }
    
    # First simulation: linear controller
    print('Running simulation with linear control...')
    env.pv_node_reset()
    state = env.reset0()
    episode_reward = 0
    last_action = np.zeros((agent_num, 1))
    action_list = []
    state_list = []
    state_list.append(state)
    
    for step in tqdm(range(p.shape[0])):
        if step == topology_change_step:
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        
        state1 = np.asarray(state-env.vmax)
        state2 = np.asarray(env.vmin-state)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((agent_num,1))
        
        action = (last_action - 10*d_v)
        
        last_action = np.copy(action)

        next_state, topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        action_list.append(action)
        state_list.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    results['no_control'] = {
        'action_list': action_list,
        'state_list': np.array(state_list),
        'p': p[:len(action_list)],
        'q': q[:len(action_list)],
        'pv_p': pv_p[:len(action_list)]
    }
    
    # Second simulation: using RLCFT
    print('\nRunning simulation with RLCFT control...')
    env.pv_node_reset()
    state = env.reset0()
    topology = torch.cuda.FloatTensor(np.array(env.topology_init)).unsqueeze(0)
    last_action = np.zeros((agent_num, 1))
    action_list_RLCFT = []
    state_list_RLCFT = []
    state_list_RLCFT.append(state)
    
    for step in tqdm(range(p.shape[0])):
        if step == topology_change_step:
            result = env.pv_node_change(action='add', bus_id=9, p_mw=pv_p[step]*0.35)
            print(f"PV node addition result: {result}")
        
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
            action_agent = 1.0* action_agent.detach().cpu().numpy()[0]
            action.append(action_agent)

        # PI policy    
        action = last_action - np.asarray(action)

        # Execute action
        next_state, topology, reward, done = env.step_load(action, p[step], q[step], pv_p[step])
        topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
    
        action_list_RLCFT.append(action)
        state_list_RLCFT.append(next_state)
        last_action = np.copy(action)
        state = next_state
    
    results['rlcft_control'] = {
        'action_list': action_list_RLCFT,
        'state_list': np.array(state_list_RLCFT)
    }
    
    print("\nSimulation completed!")
    return results

# Run simulation and save results
simulation_data_cp = run_simulation_safe(load_p, load_q, PV_p, topology_change_step=7000)